# EDSS school-year core mart validation

## tl;dr

This notebook verifies that `analysis.school_year_core_2010_2022` has exactly 24,044 unique school-year keys, retains 521 keys without 0101 coverage as missing rather than zero, and aggregates 710 multiple-campus keys without join expansion.

## Context & Methods

The mart starts from the committed school-year bridge population and left-joins an aggregation of panel 0101 at `(_panel_year, 개방ID)` grain. This notebook checks the materialized table, committed data dictionary, and build audit.

### Key Assumptions

- The analysis period is 2010–2022; the incompatible 2023–2024 employment schema is outside this mart.
- Multiple campus rows are additive for the 12 selected metrics.
- A missing 0101 match is unknown, not zero.
- All downstream panel joins must first aggregate their native grain to the mart key.

In [1]:
from pathlib import Path
import csv
import json
import duckdb

repo_root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
database_path = repo_root / 'data/processed/edss/restricted/edss_all.duckdb'
audit_path = repo_root / 'data/metadata/edss_duckdb_build.json'
dictionary_path = repo_root / 'data/metadata/edss_school_year_core_data_dictionary.csv'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
dictionary = list(csv.DictReader(dictionary_path.open(encoding='utf-8-sig', newline='')))
mart_audit = audit['school_year_core_mart']
con = duckdb.connect(str(database_path), read_only=True)
print({'duckdb_version': duckdb.__version__, 'table': mart_audit['table'], 'database_sha256': audit['database']['sha256']})

{'duckdb_version': '1.4.1', 'table': 'analysis.school_year_core_2010_2022', 'database_sha256': 'b4b5e3d748094b5a8209a3a2ba9471dc3cfb53087ff44620c55fc3748762d101'}


## Results

First, validate the mart grain, coverage, campus aggregation count, period, and 0101 source-row accounting.

In [2]:
stats = con.execute("""
    SELECT
        count(*),
        count(DISTINCT (_panel_year, 개방ID)),
        count(*) FILTER (WHERE coalesce(_panel_year, '') = '' OR coalesce(개방ID, '') = ''),
        count(*) FILTER (WHERE _0101_exists = 'true'),
        count(*) FILTER (WHERE _0101_exists = 'false'),
        count(*) FILTER (WHERE campus_scope = 'multiple_campuses'),
        min(_panel_year),
        max(_panel_year),
        sum(coalesce(metric_source_row_count, 0))
    FROM analysis.school_year_core_2010_2022
""").fetchone()
expected = (24044, 24044, 0, 23523, 521, 710, '2010', '2022', 24304)
assert stats == expected, (stats, expected)
assert mart_audit['duplicate_key_count'] == 0
assert mart_audit['join_expansion_count'] == 0
assert mart_audit['aggregated_0101_key_count'] == 23523
assert mart_audit['accounted_0101_source_row_count'] == 24304
print(stats)

(24044, 24044, 0, 23523, 521, 710, '2010', '2022', 24304)


The committed dictionary must cover every output column in exact order and with the physical DuckDB type.

In [3]:
schema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'analysis'
      AND table_name = 'school_year_core_2010_2022'
    ORDER BY ordinal_position
""").fetchall()
expected_schema = [(row['column_name'], row['data_type']) for row in dictionary]
assert schema == expected_schema, (schema, expected_schema)
assert len(schema) == 31
print({'columns': len(schema), 'dictionary_rows': len(dictionary)})

{'columns': 31, 'dictionary_rows': 31}


Metric QA requires numeric nonnegative source values, exact source-to-mart sum reconciliation, correct missingness, and female-subset counts no greater than totals.

In [4]:
metric_validation = mart_audit['metric_validation']
assert len(metric_validation) == 12
assert all(item['invalid_nonblank_row_count'] == 0 for item in metric_validation.values())
assert all(item['negative_row_count'] == 0 for item in metric_validation.values())
assert all(item['sum_reconciles'] is True for item in metric_validation.values())
assert mart_audit['matched_rows_with_missing_metrics'] == 0
assert mart_audit['unmatched_rows_with_populated_metrics'] == 0
assert all(value == 0 for value in mart_audit['subset_violations'].values())
assert mart_audit['bridge_left_join_output_row_count'] == 24044
print({'metrics': len(metric_validation), 'subset_checks': len(mart_audit['subset_violations']), 'join_expansion': mart_audit['join_expansion_count']})

{'metrics': 12, 'subset_checks': 5, 'join_expansion': 0}


Annual counts below make the population and 0101 coverage visible without exposing row-level records.

In [5]:
year_summary = con.execute("""
    SELECT
        _panel_year,
        count(*) AS school_year_keys,
        count(*) FILTER (WHERE _0101_exists = 'true') AS matched_0101,
        count(*) FILTER (WHERE _0101_exists = 'false') AS unmatched_0101,
        count(*) FILTER (WHERE campus_scope = 'multiple_campuses') AS multiple_campuses
    FROM analysis.school_year_core_2010_2022
    GROUP BY _panel_year
    ORDER BY _panel_year
""").fetchall()
assert len(year_summary) == 13
assert sum(row[1] for row in year_summary) == 24044
for row in year_summary:
    print(row)

('2010', 1675, 1672, 3, 54)
('2011', 1710, 1694, 16, 49)
('2012', 1785, 1764, 21, 57)
('2013', 1820, 1789, 31, 54)
('2014', 1846, 1808, 38, 63)
('2015', 1880, 1853, 27, 63)
('2016', 1891, 1834, 57, 56)
('2017', 1900, 1846, 54, 52)
('2018', 1907, 1866, 41, 51)
('2019', 1908, 1841, 67, 52)
('2020', 1904, 1844, 60, 51)
('2021', 1919, 1851, 68, 54)
('2022', 1899, 1861, 38, 54)


In [6]:
con.close()
print('school-year core mart validation complete')

school-year core mart validation complete


## Takeaways

- The mart contains 24,044 unique 2010–2022 school-year keys.
- 23,523 keys have 0101 coverage; 521 unmatched keys remain present with `NULL` metrics.
- 24,304 0101 rows collapse to school-year grain, including 710 multiple-campus keys.
- All 12 metric sums reconcile exactly, all subset checks pass, and the bridge join adds no rows.